# DAD Protocol — Docking Analysis of Dipeptides

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hyunho-song09/dad-protocol/blob/main/DAD_protocol.ipynb)

**Two-Phase Workflow**

| Phase | Steps | Input | Output |
|-------|-------|-------|--------|
| **Phase A** — Structure prep (run once) | §0–4 | Multi-FASTA | PDB + pocket cache, `structure_registry.tsv` |
| **Phase B** — Selective ligand scoring | §5–10 | Multi-SMILES + protein/ligand selection | `docking_master.csv`, heatmap, py3Dmol view |

**Quick Start:**
1. `Runtime → Run all` (first run installs condacolab; kernel restarts — run all again)
2. **§1** Paste multi-FASTA sequences
3. **§2** Choose `STRUCTURE_MODE`: `colabfold_af2` (default, ~10–20 min/protein on GPU) or set `af3_results` to ingest AlphaFold Server CIF files
4. **§3** P2Rank pocket detection runs automatically on all proteins
5. **§4** Review `structure_registry.tsv` and pLDDT bars
6. **§5** Paste SMILES (one `name:SMILES` per line)
7. **§6** Select protein × ligand subset with the widget checkboxes, click **Run docking on selection**
8. **§8** Export `docking_master.csv`

Re-running Phase B with new SMILES does **not** re-run Phase A (PDB cache hit).

In [ ]:
#@title 0. Setup Environment {display-mode: "form"}
#@markdown Installs condacolab (first run), mamba packages, and Python dependencies.
#@markdown If condacolab installs, Colab restarts the kernel. Run all cells again after restart.
#@markdown **Output:** progress bar, package check, optional GNINA check.

import sys, os, subprocess, json, hashlib, shutil, time
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional, List, Dict

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

class DADProgress:
    def __init__(self, label: str, total: int):
        self.label = label; self.total = total; self.n = 0
        try:
            from tqdm.notebook import tqdm
            self._bar = tqdm(total=total, desc=label, leave=True)
        except Exception:
            self._bar = None
            print(f'[{label}] 0/{total}')
    def update(self, msg=''):
        self.n += 1
        if self._bar:
            self._bar.set_postfix_str(str(msg)[:60])
            self._bar.update(1)
        else:
            print(f'[{self.label}] {self.n}/{self.total} {msg}')
    def close(self):
        if self._bar: self._bar.close()

def dad_progress(label: str, total: int) -> DADProgress:
    return DADProgress(label, total)

def run_cmd(cmd, desc=''):
    if desc: print(f'  {desc}...')
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-500:]); print(r.stderr[-500:])
        raise RuntimeError(f'{desc} failed (exit {r.returncode})')
    return r

# ── condacolab install (codex fix: RESTART_REQUESTED flag, no SystemExit) ────
RESTART_REQUESTED = False
if IN_COLAB and shutil.which('mamba') is None:
    p = dad_progress('Setup Environment', 2)
    p.update('install condacolab')
    run_cmd([sys.executable, '-m', 'pip', 'install', '-q', 'condacolab'], 'Install condacolab')
    import condacolab
    p.update('condacolab.install → kernel restart')
    condacolab.install()
    RESTART_REQUESTED = True
    print('\n>>> condacolab installed. Colab is restarting the kernel.')
    print('>>> After restart, click Runtime → Run all to continue setup.')

if not RESTART_REQUESTED:
    p = dad_progress('Setup Environment', 9)

    # mamba packages
    if IN_COLAB and shutil.which('mamba'):
        for pkg in ['rdkit', 'biopython', 'tqdm', 'openbabel-wheel']:
            try:
                run_cmd(['mamba', 'install', '-y', '-q', '-c', 'conda-forge', pkg])
            except RuntimeError:
                pass
    p.update('mamba packages')

    # pip packages
    run_cmd([sys.executable, '-m', 'pip', 'install', '-q',
             'pandas', 'numpy', 'matplotlib', 'scipy', 'scikit-learn',
             'py3Dmol', 'ipywidgets'], 'pip packages')
    p.update('pip packages')

    # P2Rank
    P2RANK_DIR = Path('/content/p2rank') if IN_COLAB else Path.home() / '.p2rank'
    P2RANK_BIN = P2RANK_DIR / 'prank'
    if not P2RANK_BIN.exists():
        P2RANK_URL = 'https://github.com/rdk/p2rank/releases/download/2.4.2/p2rank_2.4.2.tar.gz'
        try:
            run_cmd(['wget', '-q', P2RANK_URL, '-O', '/tmp/p2rank.tar.gz'])
            run_cmd(['tar', '-xzf', '/tmp/p2rank.tar.gz', '-C', '/tmp'])
            shutil.move('/tmp/p2rank_2.4.2', str(P2RANK_DIR))
        except Exception as e:
            print(f'P2Rank download skipped: {e}')
    p.update('P2Rank')

    # GNINA
    GNINA_BIN = Path('/usr/local/bin/gnina')
    GNINA_URL = 'https://github.com/gnina/gnina/releases/download/v1.3.2/gnina.1.3.2'
    GNINA_VERSION_EXPECTED = 'gnina v1.3.2'
    if not GNINA_BIN.exists() or GNINA_VERSION_EXPECTED not in subprocess.run(
            [str(GNINA_BIN), '--version'], capture_output=True, text=True).stdout:
        try:
            run_cmd(['wget', '-q', GNINA_URL, '-O', str(GNINA_BIN)])
            GNINA_BIN.chmod(0o755)
        except Exception as e:
            print(f'GNINA download skipped: {e}')
    p.update('GNINA')

    # DeepTMHMM
    try:
        run_cmd([sys.executable, '-m', 'pip', 'install', '-q', 'pybiolib'], 'DeepTMHMM (biolib)')
    except Exception:
        pass
    p.update('DeepTMHMM')

    # Work directory
    DAD_ROOT = Path(os.environ.get('DAD_ROOT', '/content/DAD'))
    WORK_DIR = DAD_ROOT / 'work'
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    p.update('work dir')

    # phase_a + phase_b cache dirs
    PHASE_A_DIR = WORK_DIR / 'phase_a'
    PHASE_B_DIR = WORK_DIR / 'phase_b'
    for d in [
        PHASE_A_DIR / 'colabfold_out',
        PHASE_A_DIR / 'structure_cache',
        PHASE_A_DIR / 'pocket_cache',
        PHASE_B_DIR / 'ligands',
        PHASE_B_DIR / 'runs',
    ]:
        d.mkdir(parents=True, exist_ok=True)
    p.update('cache dirs')

    p.close()
    print('Setup complete. WORK_DIR:', WORK_DIR)

---
## Phase A — Structure Preparation
*Run once per protein set. Re-running with the same sequences uses the cache (no GPU time wasted).*

In [ ]:
#@title 1. Phase A — Multi-FASTA Input {display-mode: "form"}
#@markdown Paste one or more protein sequences in FASTA format.
#@markdown Raw sequences (no `>` header) are accepted as `Protein_1`, `Protein_2`, ...
#@markdown ColabFold chain-break syntax (`sequence_A:sequence_B`) is supported for complexes.
#@markdown **Output:** validated protein records list, sequence SHA cache keys.

fasta_text = """>ProteinA
MKTAYIAKQRQISFVKSHFSRQDILDLWIYHTQGYFP
>ProteinB
GSSHHHHHHSSGLVPRGSHMNNNKQQQQ"""  #@param {type:"string"}

if 'dad_progress' not in globals():
    import sys, os, subprocess, json, hashlib, shutil, time
    from pathlib import Path
    from dataclasses import dataclass, field
    from typing import Optional, List, Dict
    IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
    class DADProgress:
        def __init__(self, label, total):
            self.label = label; self.total = total; self.n = 0
            try:
                from tqdm.notebook import tqdm
                self._bar = tqdm(total=total, desc=label, leave=True)
            except Exception:
                self._bar = None
        def update(self, msg=''):
            self.n += 1
            if self._bar: self._bar.set_postfix_str(str(msg)[:60]); self._bar.update(1)
            else: print(f'[{self.label}] {self.n}/{self.total} {msg}')
        def close(self):
            if self._bar: self._bar.close()
    def dad_progress(label, total): return DADProgress(label, total)
    DAD_ROOT = Path(os.environ.get('DAD_ROOT', '/content/DAD'))
    WORK_DIR = DAD_ROOT / 'work'
    PHASE_A_DIR = WORK_DIR / 'phase_a'
    PHASE_B_DIR = WORK_DIR / 'phase_b'
    for _d in [PHASE_A_DIR / 'structure_cache', PHASE_A_DIR / 'pocket_cache',
               PHASE_A_DIR / 'colabfold_out', PHASE_B_DIR / 'ligands', PHASE_B_DIR / 'runs']:
        _d.mkdir(parents=True, exist_ok=True)

@dataclass
class ProteinRecord:
    name: str
    sequence: str
    seq_sha: str = field(init=False)
    def __post_init__(self):
        self.seq_sha = hashlib.sha256(self.sequence.upper().encode()).hexdigest()[:12]

def parse_fasta(text: str) -> List[ProteinRecord]:
    records, current_name, current_seq = [], None, []
    for line in text.strip().splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith('>'):
            if current_name is not None and current_seq:
                records.append(ProteinRecord(current_name, ''.join(current_seq)))
            current_name = line[1:].split()[0]
            current_seq = []
        else:
            if current_name is None:
                current_name = f'Protein_{len(records) + 1}'
            current_seq.append(line)
    if current_name is not None and current_seq:
        records.append(ProteinRecord(current_name, ''.join(current_seq)))
    return records

p = dad_progress('Phase A Input', 3)
protein_records = parse_fasta(fasta_text)
p.update('parse FASTA')

assert len(protein_records) > 0, 'No protein sequences found in fasta_text.'
names = [r.name for r in protein_records]
assert len(names) == len(set(names)), f'Duplicate protein names: {[n for n in names if names.count(n) > 1]}'
p.update('validate')

input_fasta = PHASE_A_DIR / 'input_sequences.fasta'
input_fasta.write_text('\n'.join(f'>{r.name}\n{r.sequence}' for r in protein_records) + '\n')
p.update('write FASTA')
p.close()

print(f'Found {len(protein_records)} protein(s):')
for r in protein_records:
    print(f'  {r.name}: {len(r.sequence)} aa, sha={r.seq_sha}')

In [ ]:
#@title 2. Phase A — Structure Prediction {display-mode: "form"}
#@markdown **STRUCTURE_MODE options:**
#@markdown - `colabfold_af2` (DEFAULT): run `colabfold_batch` on GPU with multi-FASTA.
#@markdown   Speed: `--num-models 1 --num-recycle 3 --msa-mode mmseqs2_uniref_env --sort-queries-by length --zip`
#@markdown - `af3_results`: ingest CIF files from AlphaFold Server (alphafoldserver.com) results.
#@markdown   AF3 cannot be run directly on Colab (restricted weights). External results only.
#@markdown - `esmfold_api`: ESMFold public API (<=550 aa, no GPU needed).
#@markdown - `user_pdb`: upload or specify existing PDB files.
#@markdown - `auto`: try cache -> ESMFold API -> ColabFold AF2 fallback.

STRUCTURE_MODE = "colabfold_af2"  #@param ["colabfold_af2", "af3_results", "esmfold_api", "user_pdb", "auto"]

#@markdown ---
#@markdown **ColabFold AF2 settings** (active when STRUCTURE_MODE is `colabfold_af2` or `auto`):
AF2_FAST_PREVIEW = False  #@param {type:"boolean"}
#@markdown `True` adds `--num-relax 0 --templates 0` for ~2x speed (lower quality).
AF2_NUM_MODELS = 1  #@param [1, 5] {type:"raw"}
AF2_NUM_RECYCLES = 3  #@param [1, 3, 6] {type:"raw"}
AF2_RELAX_TOP_N = 0  #@param [0, 1] {type:"raw"}
#@markdown `AF2_RELAX_TOP_N > 0` runs AMBER relaxation (installs OpenMM/PDBFixer via mamba).
ALLOW_COLABFOLD_FALLBACK = True  #@param {type:"boolean"}

#@markdown ---
#@markdown **AF3 result ingest** (active when STRUCTURE_MODE is `af3_results`):
AF3_RESULTS_PATH = ""  #@param {type:"string"}
#@markdown Path to the AlphaFold Server results folder. Leave empty to upload a ZIP in Colab.
AF3_USE_TOP_RANKED = True  #@param {type:"boolean"}
#@markdown `True` = use model_0 only; `False` = load all 5 models and pick highest ranking_score.

import re

if 'structure_registry' not in globals():
    structure_registry: Dict[str, dict] = {}

STRUCT_CACHE = PHASE_A_DIR / 'structure_cache'

def _canonical_pdb_path(record: ProteinRecord) -> Path:
    return STRUCT_CACHE / f'{record.seq_sha}_{record.name}.pdb'

def _register(record: ProteinRecord, pdb_path: Path, source: str, plddt_mean: float = 0.0):
    structure_registry[record.name] = {
        'name': record.name, 'seq_sha': record.seq_sha,
        'pdb_path': str(pdb_path), 'source': source, 'plddt_mean': round(plddt_mean, 2)
    }

# ── pLDDT visualization helper ───────────────────────────────────────────────
def _plot_plddt_bar(name: str, plddt_vals: list):
    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(max(6, len(plddt_vals) / 10), 3))
        ax.bar(range(len(plddt_vals)), plddt_vals, color='steelblue', linewidth=0)
        ax.axhline(70, color='orange', linewidth=1, linestyle='--', label='70')
        ax.axhline(90, color='green',  linewidth=1, linestyle='--', label='90')
        ax.set_xlabel('Residue'); ax.set_ylabel('pLDDT')
        ax.set_title(f'{name} — per-residue pLDDT'); ax.legend(fontsize=8); ax.set_ylim(0, 100)
        plt.tight_layout(); plt.show()
    except Exception as e:
        print(f'pLDDT plot skipped: {e}')

# ── CIF -> PDB (Bio.PDB primary, gemmi fallback) ─────────────────────────────
def _cif_to_pdb(cif_path: Path, out_pdb: Path) -> Path:
    try:
        from Bio.PDB import MMCIFParser, PDBIO
        structure = MMCIFParser(QUIET=True).get_structure('s', str(cif_path))
        io = PDBIO(); io.set_structure(structure); io.save(str(out_pdb))
        return out_pdb
    except Exception:
        pass
    if shutil.which('gemmi'):
        subprocess.run(['gemmi', 'convert', str(cif_path), str(out_pdb)], check=True)
        return out_pdb
    shutil.copy2(str(cif_path), str(out_pdb))
    return out_pdb

def _parse_af3_confidences(conf_path: Path) -> dict:
    d = json.load(open(conf_path))
    atom_plddt = d.get('atom_plddts') or d.get('plddt') or []
    mean_plddt = sum(atom_plddt) / len(atom_plddt) if atom_plddt else 0.0
    return {
        'iptm': d.get('iptm', d.get('ipTM', None)),
        'ptm':  d.get('ptm',  d.get('pTM', None)),
        'mean_plddt': round(mean_plddt, 2),
        'ranking_score': d.get('ranking_score', None),
        'atom_plddt': atom_plddt,
    }

def _atom_plddt_to_residue(atom_plddt: list, cif_path: Path) -> list:
    try:
        from Bio.PDB import MMCIFParser
        structure = MMCIFParser(QUIET=True).get_structure('s', str(cif_path))
        residues = [len(list(res.get_atoms())) for chain in structure.get_chains()
                    for res in chain.get_residues()]
        res_plddt, idx = [], 0
        for n in residues:
            chunk = atom_plddt[idx:idx + n]
            res_plddt.append(sum(chunk) / len(chunk) if chunk else 0.0)
            idx += n
        return res_plddt
    except Exception:
        return atom_plddt

# ── AF3 ingest ───────────────────────────────────────────────────────────────
def _ingest_af3(record: ProteinRecord, af3_root: Path):
    candidates = [af3_root / record.name, af3_root / record.name.lower(), af3_root]
    af3_dir = next((d for d in candidates if d.is_dir() and list(d.glob('*.cif'))), None)
    if af3_dir is None:
        print(f'  AF3: no CIF folder for {record.name} under {af3_root}')
        return
    cif_files = sorted(af3_dir.glob('*_model_*.cif')) or sorted(af3_dir.glob('*.cif'))
    if not cif_files:
        print(f'  AF3: no CIF files for {record.name}')
        return
    if AF3_USE_TOP_RANKED:
        chosen = next((f for f in cif_files if '_model_0' in f.name), cif_files[0])
        conf_stem = chosen.stem.replace('_model_0', '_summary_confidences_0')
        cp = chosen.parent / (conf_stem + '.json')
        confs = _parse_af3_confidences(cp) if cp.exists() else {}
    else:
        best_score, chosen, confs = -1.0, cif_files[0], {}
        for f in cif_files:
            m = re.search(r'_model_(\d+)', f.name)
            idx = m.group(1) if m else '0'
            cp = f.parent / (re.sub(r'_model_\d+', f'_summary_confidences_{idx}', f.stem) + '.json')
            c = _parse_af3_confidences(cp) if cp.exists() else {}
            if (c.get('ranking_score') or 0.0) > best_score:
                best_score, chosen, confs = c.get('ranking_score', 0.0), f, c
    out_pdb = _canonical_pdb_path(record)
    _cif_to_pdb(chosen, out_pdb)
    mean_plddt = confs.get('mean_plddt', 0.0)
    print(f'  AF3 {record.name}: ipTM={confs.get("iptm","N/A")}, pTM={confs.get("ptm","N/A")}, '
          f'mean_pLDDT={mean_plddt}')
    _register(record, out_pdb, 'af3_results', mean_plddt)
    atom_pl = confs.get('atom_plddt', [])
    if atom_pl:
        _plot_plddt_bar(record.name, _atom_plddt_to_residue(atom_pl, chosen))

# ── ColabFold AF2 helpers (codex fixes preserved) ────────────────────────────
def patch_tensorflow_for_colabfold():
    """Codex fix: remove conflicting TF .so files before ColabFold import on Python 3.12."""
    try:
        import tensorflow as tf
        tf_dir = Path(tf.__file__).parent
        for so in tf_dir.rglob('*pywrap_tensorflow*.so'):
            bak = so.with_suffix('.so.bak')
            if not bak.exists():
                shutil.move(str(so), str(bak))
    except Exception:
        pass

def _colabfold_alphafold_ok() -> bool:
    try:
        import importlib; importlib.import_module('alphafold'); return True
    except ImportError:
        return False

def ensure_colabfold_ready():
    if shutil.which('colabfold_batch') and _colabfold_alphafold_ok():
        return
    print('  Installing ColabFold + AlphaFold extras...')
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q', '--no-warn-conflicts',
        'colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold'
    ], check=True)
    patch_tensorflow_for_colabfold()

def ensure_amber_ready():
    try:
        import openmm  # noqa
    except ImportError:
        if shutil.which('mamba'):
            subprocess.run(['mamba', 'install', '-y', '-q', '-c', 'conda-forge',
                            'openmm', 'pdbfixer'], check=False)
        else:
            raise RuntimeError('AF2_RELAX_TOP_N requires OpenMM/PDBFixer. Set AF2_RELAX_TOP_N=0 or install mamba.')

def apply_af2_preset():
    return int(AF2_NUM_RECYCLES), int(AF2_NUM_MODELS), 1, int(AF2_RELAX_TOP_N)

# ── ESMFold API (codex fix: 413 fallback) ────────────────────────────────────
def predict_with_esmfold_api(record: ProteinRecord) -> Optional[Path]:
    import urllib.request, urllib.error
    url = 'https://api.esmatlas.com/foldSequence/v1/pdb/'
    try:
        req = urllib.request.Request(url, data=record.sequence.encode(), method='POST',
                                      headers={'Content-Type': 'application/x-www-form-urlencoded'})
        with urllib.request.urlopen(req, timeout=120) as resp:
            pdb_text = resp.read().decode()
    except urllib.error.HTTPError as exc:
        print(f'  ESMFold HTTP {exc.code} for {record.name} ({len(record.sequence)} aa)')
        return None
    except Exception as exc:
        print(f'  ESMFold error for {record.name}: {exc}')
        return None
    if 'ATOM' not in pdb_text[:400]:
        print(f'  ESMFold returned non-PDB response for {record.name}')
        return None
    out_pdb = _canonical_pdb_path(record)
    out_pdb.write_text(pdb_text)
    plddt_vals = [float(l[60:66]) for l in pdb_text.splitlines() if l.startswith('ATOM')]
    mean_plddt = sum(plddt_vals) / len(plddt_vals) if plddt_vals else 0.0
    _register(record, out_pdb, 'esmfold_api', mean_plddt)
    return out_pdb

# ── colabfold_batch CLI (multi-FASTA, speed settings from AF2_colabfold_batch_setting_plan.md)
def run_colabfold_batch(fasta_path: Path = None) -> Dict[str, Path]:
    ensure_colabfold_ready()
    num_recycles, num_models, _, relax_n = apply_af2_preset()
    if relax_n > 0:
        ensure_amber_ready()
    if fasta_path is None:
        fasta_path = input_fasta
    out_dir = PHASE_A_DIR / 'colabfold_out'
    cmd = [
        'colabfold_batch',
        str(fasta_path), str(out_dir),
        '--model-type', 'auto',
        '--msa-mode', 'mmseqs2_uniref_env',
        '--num-models', str(num_models),
        '--num-recycle', str(num_recycles),
        '--sort-queries-by', 'length',
        '--zip',
    ]
    if AF2_FAST_PREVIEW:
        cmd += ['--num-relax', '0', '--templates', '0']
    elif relax_n > 0:
        cmd += ['--num-relax', str(relax_n)]
    print('  colabfold_batch:', ' '.join(cmd))
    log_path = PHASE_A_DIR / 'colabfold.log'
    with open(log_path, 'w') as lf:
        r = subprocess.run(cmd, stdout=lf, stderr=subprocess.STDOUT)
    if r.returncode != 0:
        raise RuntimeError(f'colabfold_batch failed. See {log_path}')
    # collect rank_001 PDBs
    name_to_pdb: Dict[str, Path] = {}
    for record in protein_records:
        pdbs = (sorted(out_dir.rglob(f'{record.name}*rank_001*.pdb')) +
                sorted(out_dir.rglob(f'{record.name}*rank_1*.pdb')))
        if not pdbs:
            print(f'  WARNING: no rank_001 PDB for {record.name}')
            continue
        dst = _canonical_pdb_path(record)
        shutil.copy2(str(sorted(pdbs)[0]), str(dst))
        plddt_vals = [float(l[60:66]) for l in dst.read_text().splitlines() if l.startswith('ATOM')]
        mean_plddt = sum(plddt_vals) / len(plddt_vals) if plddt_vals else 0.0
        _register(record, dst, 'colabfold_af2', mean_plddt)
        _plot_plddt_bar(record.name, plddt_vals)
        name_to_pdb[record.name] = dst
    return name_to_pdb

# ── Main dispatch ─────────────────────────────────────────────────────────────
p = dad_progress('Phase A Structure', max(1, len(protein_records) + 2))

if STRUCTURE_MODE == 'af3_results':
    if AF3_RESULTS_PATH.strip():
        af3_root = Path(AF3_RESULTS_PATH.strip())
    elif IN_COLAB:
        print('AF3_RESULTS_PATH empty — upload AF3 results ZIP:')
        from google.colab import files
        import zipfile
        uploaded = files.upload()
        af3_upload_dir = Path('/content/af3_upload'); af3_upload_dir.mkdir(exist_ok=True)
        for fname, data in uploaded.items():
            zp = Path('/tmp') / fname; zp.write_bytes(data)
            with zipfile.ZipFile(zp) as zf: zf.extractall(af3_upload_dir)
        af3_root = af3_upload_dir
    else:
        raise RuntimeError('Set AF3_RESULTS_PATH to the folder with AF3 CIF files.')
    p.update('AF3 root')
    for record in protein_records:
        if _canonical_pdb_path(record).exists() and record.name in structure_registry:
            print(f'  Cache hit: {record.name}'); p.update(f'cache {record.name}'); continue
        _ingest_af3(record, af3_root)
        p.update(f'AF3 {record.name}')

elif STRUCTURE_MODE == 'colabfold_af2':
    missing = [r for r in protein_records
               if not _canonical_pdb_path(r).exists() or r.name not in structure_registry]
    if not missing:
        print('All proteins in cache. Skipping colabfold_batch.')
        [p.update(f'cache {r.name}') for r in protein_records]
    else:
        print(f'  colabfold_batch for {len(missing)} protein(s)...')
        if len(missing) < len(protein_records):
            tmp = PHASE_A_DIR / 'input_missing.fasta'
            tmp.write_text('\n'.join(f'>{r.name}\n{r.sequence}' for r in missing) + '\n')
            run_colabfold_batch(tmp)
        else:
            run_colabfold_batch()
        for record in protein_records:
            p.update(f'done {record.name}')

elif STRUCTURE_MODE in ('esmfold_api', 'auto'):
    for record in protein_records:
        if _canonical_pdb_path(record).exists() and record.name in structure_registry:
            print(f'  Cache hit: {record.name}'); p.update(f'cache {record.name}'); continue
        result_pdb = predict_with_esmfold_api(record)
        if result_pdb is None and ALLOW_COLABFOLD_FALLBACK:
            print(f'  ESMFold failed -> ColabFold AF2 fallback for {record.name}')
            tmp = PHASE_A_DIR / f'tmp_{record.name}.fasta'
            tmp.write_text(f'>{record.name}\n{record.sequence}\n')
            run_colabfold_batch(tmp)
        elif result_pdb is None and STRUCTURE_MODE == 'esmfold_api':
            raise RuntimeError(
                f'ESMFold failed for {record.name}. '
                f'Set ALLOW_COLABFOLD_FALLBACK=True or change STRUCTURE_MODE to auto.')
        p.update(f'ESMFold {record.name}')

elif STRUCTURE_MODE == 'user_pdb':
    for record in protein_records:
        cached = _canonical_pdb_path(record)
        if cached.exists():
            plddt_vals = [float(l[60:66]) for l in cached.read_text().splitlines() if l.startswith('ATOM')]
            mean_plddt = sum(plddt_vals) / len(plddt_vals) if plddt_vals else 0.0
            _register(record, cached, 'user_pdb', mean_plddt)
        elif IN_COLAB:
            print(f'Upload PDB for {record.name}:')
            from google.colab import files
            up = files.upload()
            for fname, data in up.items(): cached.write_bytes(data)
            _register(record, cached, 'user_pdb')
        else:
            raise RuntimeError(f'PDB not found for {record.name}: {cached}')
        p.update(f'user_pdb {record.name}')

p.update('done'); p.close()
print(f'\nstructure_registry ({len(structure_registry)} proteins):')
for name, row in structure_registry.items():
    print(f'  {name}: {row["source"]}, pLDDT={row["plddt_mean"]}, pdb={row["pdb_path"]}')

In [ ]:
#@title 3. Phase A — Pocket Detection {display-mode: "form"}
#@markdown Runs P2Rank on every protein in structure_registry.
#@markdown **Output:** pocket_cache CSVs, pocket_registry with box center coordinates.

POCKET_RANK = 1  #@param {type:"integer"}

POCKET_CACHE = PHASE_A_DIR / 'pocket_cache'

def run_p2rank(name: str, pdb_path: Path) -> Optional[Path]:
    out_dir = POCKET_CACHE / name; out_dir.mkdir(parents=True, exist_ok=True)
    csv_dest = out_dir / 'predictions.csv'
    if csv_dest.exists():
        return csv_dest
    p2rank_bin = (
        P2RANK_DIR / 'prank' if (P2RANK_DIR / 'prank').exists()
        else Path(shutil.which('prank') or '')
    )
    if not p2rank_bin.exists():
        print(f'  P2Rank not found - pocket detection skipped for {name}')
        return None
    r = subprocess.run([str(p2rank_bin), 'predict', '-f', str(pdb_path),
                        '-o', str(out_dir), '-threads', '2'],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(f'  P2Rank failed for {name}: {r.stderr[-200:]}')
        return None
    found = list(out_dir.rglob('*_predictions.csv')) + list(out_dir.rglob('predictions.csv'))
    if not found:
        return None
    shutil.copy2(str(found[0]), str(csv_dest))
    return csv_dest

def read_pocket(csv_path: Path, rank: int = 1):
    import pandas as pd
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    if len(df) == 0: return None
    return df.iloc[min(rank - 1, len(df) - 1)]

if 'pocket_registry' not in globals():
    pocket_registry: Dict[str, dict] = {}

p = dad_progress('Phase A Pocket', max(1, len(structure_registry)))
for name, row in structure_registry.items():
    csv_path = run_p2rank(name, Path(row['pdb_path']))
    if csv_path is not None:
        pr = read_pocket(csv_path, POCKET_RANK)
        if pr is not None:
            pocket_registry[name] = {
                'csv': str(csv_path),
                'center_x': float(pr.get('center_x', pr.get('x', 0))),
                'center_y': float(pr.get('center_y', pr.get('y', 0))),
                'center_z': float(pr.get('center_z', pr.get('z', 0))),
                'score': float(pr.get('score', pr.get('probability', 0))),
            }
            print(f'  {name}: score={pocket_registry[name]["score"]:.3f}')
    p.update(name)
p.close()
print(f'Pocket detection done: {len(pocket_registry)}/{len(structure_registry)} proteins.')

In [ ]:
#@title 4. Phase A — Summary {display-mode: "form"}
#@markdown Writes `structure_registry.tsv` and shows per-protein pLDDT bar charts.
#@markdown Phase A is complete after this cell. Proceed to Phase B (SS5).

import pandas as pd, matplotlib.pyplot as plt

p = dad_progress('Phase A Summary', 3)

rows = []
for name, s in structure_registry.items():
    pocket = pocket_registry.get(name, {})
    rows.append({
        'protein_name': name, 'seq_sha': s['seq_sha'], 'source': s['source'],
        'plddt_mean': s['plddt_mean'], 'pdb_path': s['pdb_path'],
        'pocket_csv': pocket.get('csv', ''), 'pocket_score': pocket.get('score', ''),
        'pocket_cx': pocket.get('center_x', ''), 'pocket_cy': pocket.get('center_y', ''),
        'pocket_cz': pocket.get('center_z', ''),
    })
registry_df = pd.DataFrame(rows)
tsv_path = PHASE_A_DIR / 'structure_registry.tsv'
registry_df.to_csv(tsv_path, sep='\t', index=False)
p.update('write TSV')

try:
    from IPython.display import display as _display; _display(registry_df)
except Exception:
    print(registry_df.to_string())
p.update('display')

for name, s in structure_registry.items():
    pdb_path = Path(s['pdb_path'])
    if pdb_path.exists():
        plddt_vals = [float(l[60:66]) for l in pdb_path.read_text().splitlines() if l.startswith('ATOM')]
        if plddt_vals:
            _plot_plddt_bar(name, plddt_vals)
p.update('pLDDT plots'); p.close()

print(f'Phase A complete. structure_registry.tsv: {tsv_path}')
print('Proceed to Phase B (SS5) to enter SMILES and run selective docking.')

---
## Phase B — Selective Ligand Scoring
*Enter SMILES, select protein x ligand subsets, and run only the pairs you need.*
*Phase A results are cached — re-running Phase B never re-runs structure prediction.*

In [ ]:
#@title 5. Phase B — Multi-SMILES Input {display-mode: "form"}
#@markdown One entry per line: `name:SMILES` or bare SMILES (auto-named Ligand_1, Ligand_2, ...).
#@markdown **Output:** ligand_registry with SHA-keyed SDF cache.

smiles_text = """Ala-Ile:CC(N)C(=O)NC(CC(C)C)C(=O)O
Gly-Val:NCC(=O)NC(C(C)C)C(=O)O"""  #@param {type:"string"}

LIGAND_DIR = PHASE_B_DIR / 'ligands'
LIGAND_DIR.mkdir(parents=True, exist_ok=True)

@dataclass
class LigandRecord:
    name: str
    smiles: str
    smiles_sha: str = field(init=False)
    sdf_path: Optional[Path] = field(default=None, init=False)
    def __post_init__(self):
        self.smiles_sha = hashlib.sha256(self.smiles.encode()).hexdigest()[:12]

def parse_smiles(text: str) -> List[LigandRecord]:
    records, idx = [], 1
    for line in text.strip().splitlines():
        line = line.strip()
        if not line or line.startswith('#'): continue
        name, smiles = (line.split(':', 1) if ':' in line else (f'Ligand_{idx}', line))
        records.append(LigandRecord(name.strip(), smiles.strip()))
        idx += 1
    return records

def smiles_to_sdf(record: LigandRecord) -> Optional[Path]:
    out_sdf = LIGAND_DIR / f'{record.smiles_sha}_{record.name}.sdf'
    if out_sdf.exists(): return out_sdf
    try:
        from rdkit import Chem; from rdkit.Chem import AllChem
        mol = Chem.MolFromSmiles(record.smiles)
        if mol is None: raise ValueError(f'RDKit cannot parse: {record.smiles}')
        mol = Chem.AddHs(mol)
        AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())
        AllChem.MMFFOptimizeMolecule(mol)
        w = Chem.SDWriter(str(out_sdf)); mol.SetProp('_Name', record.name); w.write(mol); w.close()
        return out_sdf
    except Exception as e:
        if shutil.which('obabel'):
            r = subprocess.run(['obabel', f'-:{record.smiles}', '--gen3d', '-osdf', '-O', str(out_sdf)],
                               capture_output=True, text=True)
            if r.returncode == 0 and out_sdf.exists(): return out_sdf
        print(f'  3D gen failed for {record.name}: {e}')
        return None

p = dad_progress('Phase B Input', 3)
ligand_records = parse_smiles(smiles_text); p.update('parse SMILES')
assert len(ligand_records) > 0, 'No SMILES found.'
lnames = [r.name for r in ligand_records]
assert len(lnames) == len(set(lnames)), 'Duplicate ligand names.'
p.update('validate')

ligand_registry: Dict[str, LigandRecord] = {}
for rec in ligand_records:
    rec.sdf_path = smiles_to_sdf(rec)
    ligand_registry[rec.name] = rec
    print(f'  {rec.name}: sha={rec.smiles_sha}, sdf={"OK" if rec.sdf_path else "FAILED"}')
p.update('3D structures'); p.close()
print(f'{len(ligand_records)} ligand(s) prepared.')

In [ ]:
#@title 6. Phase B — Selection UI {display-mode: "form"}
#@markdown Select proteins and ligands to dock using the multi-select widgets.
#@markdown Click **Select all proteins** or **Select all ligands** to include everything.
#@markdown Then proceed to SS7 and call `_run_selected_docking()` or click **Run docking**.

try:
    import ipywidgets as widgets
    from IPython.display import display as _display

    _prot_opts = list(structure_registry.keys())
    _lig_opts  = list(ligand_registry.keys())

    protein_select = widgets.SelectMultiple(
        options=_prot_opts, value=_prot_opts[:1] if _prot_opts else [],
        description='Proteins:', rows=min(8, max(2, len(_prot_opts))),
        layout=widgets.Layout(width='320px')
    )
    ligand_select = widgets.SelectMultiple(
        options=_lig_opts, value=_lig_opts[:1] if _lig_opts else [],
        description='Ligands:', rows=min(8, max(2, len(_lig_opts))),
        layout=widgets.Layout(width='320px')
    )
    all_prot_btn = widgets.Button(description='Select all proteins',
                                   layout=widgets.Layout(width='190px'))
    all_lig_btn  = widgets.Button(description='Select all ligands',
                                   layout=widgets.Layout(width='190px'))
    run_btn      = widgets.Button(description='Run docking on selection',
                                   button_style='primary', layout=widgets.Layout(width='220px'))
    out_widget   = widgets.Output()

    all_prot_btn.on_click(lambda b: setattr(protein_select, 'value', list(protein_select.options)))
    all_lig_btn.on_click( lambda b: setattr(ligand_select,  'value', list(ligand_select.options)))

    _ui_ready = True
    _display(widgets.HBox([protein_select, ligand_select]))
    _display(widgets.HBox([all_prot_btn, all_lig_btn, run_btn]))
    _display(out_widget)
    print('Selection UI ready. Use widgets above, then run SS7.')

except ImportError:
    print('ipywidgets not available - using all proteins x all ligands.')
    _ui_ready = False
    protein_select = type('_S', (), {'value': list(structure_registry.keys())})()
    ligand_select  = type('_S', (), {'value': list(ligand_registry.keys())})()

In [ ]:
#@title 7. Phase B — Combinatorial GNINA Dispatcher {display-mode: "form"}
#@markdown Runs GNINA for selected protein x ligand cross-product only.
#@markdown Already-completed pairs (cache hit on case_id) are skipped.
#@markdown **Output:** per-pair results in `phase_b/runs/<case_id>/`, `docking_results` list.

GNINA_BIN    = Path('/usr/local/bin/gnina')
BOX_SIZE     = 22.0  #@param {type:"number"}
EXHAUSTIVENESS = 8   #@param {type:"integer"}

@dataclass
class DockingResult:
    case_id: str
    protein_name: str
    ligand_name: str
    vina_score: Optional[float]
    cnn_pose_score: Optional[float]
    cnn_affinity: Optional[float]
    pose_count: int
    out_sdf: Optional[str]
    status: str
    runtime_s: float = 0.0

def _parse_sdf_scores(sdf_path: Path) -> dict:
    props: dict = {'pose_count': 0}
    if not sdf_path.exists(): return props
    text = sdf_path.read_text()
    props['pose_count'] = text.count('$$$$')
    first_pose = text.split('$$$$')[0] if '$$$$' in text else text
    aliases = {'minimizedAffinity': 'vina_score', 'CNNscore': 'cnn_pose_score', 'CNNaffinity': 'cnn_affinity'}
    for tag, key in aliases.items():
        m = re.search(rf'<{tag}>\s*([\-\d.]+)', first_pose)
        if m:
            try: props[key] = float(m.group(1))
            except ValueError: pass
    return props

def run_gnina(protein_name: str, ligand_name: str) -> DockingResult:
    p_row = structure_registry.get(protein_name)
    l_rec = ligand_registry.get(ligand_name)
    if p_row is None or l_rec is None or l_rec.sdf_path is None:
        return DockingResult(f'{protein_name}_x_{ligand_name}', protein_name, ligand_name,
                             None, None, None, 0, None, 'missing_input')
    case_id = f'{p_row["seq_sha"]}_{l_rec.smiles_sha}'
    out_dir  = PHASE_B_DIR / 'runs' / case_id
    out_sdf  = out_dir / 'docked.sdf'
    if out_sdf.exists():
        props = _parse_sdf_scores(out_sdf)
        return DockingResult(case_id, protein_name, ligand_name,
                             props.get('vina_score'), props.get('cnn_pose_score'),
                             props.get('cnn_affinity'), int(props.get('pose_count', 0)),
                             str(out_sdf), 'cache_hit')
    out_dir.mkdir(parents=True, exist_ok=True)
    pocket = pocket_registry.get(protein_name, {})
    cmd = [
        str(GNINA_BIN),
        '-r', p_row['pdb_path'], '-l', str(l_rec.sdf_path), '-o', str(out_sdf),
        '--center_x', str(pocket.get('center_x', 0.0)),
        '--center_y', str(pocket.get('center_y', 0.0)),
        '--center_z', str(pocket.get('center_z', 0.0)),
        '--size_x', str(BOX_SIZE), '--size_y', str(BOX_SIZE), '--size_z', str(BOX_SIZE),
        '--exhaustiveness', str(EXHAUSTIVENESS),
        '--num_modes', '9', '--cnn_scoring', 'rescore',
    ]
    t0 = time.time()
    try:
        with open(out_dir / 'gnina.log', 'w') as lf:
            r = subprocess.run(cmd, stdout=lf, stderr=subprocess.STDOUT, timeout=600)
        runtime = time.time() - t0
        if r.returncode != 0:
            return DockingResult(case_id, protein_name, ligand_name,
                                 None, None, None, 0, None, 'gnina_error', runtime)
    except subprocess.TimeoutExpired:
        return DockingResult(case_id, protein_name, ligand_name,
                             None, None, None, 0, None, 'timeout', 600.0)
    props = _parse_sdf_scores(out_sdf)
    return DockingResult(case_id, protein_name, ligand_name,
                         props.get('vina_score'), props.get('cnn_pose_score'),
                         props.get('cnn_affinity'), int(props.get('pose_count', 0)),
                         str(out_sdf), 'ok', time.time() - t0)

def _run_selected_docking() -> list:
    selected_proteins = list(protein_select.value)
    selected_ligands  = list(ligand_select.value)
    pairs = [(pn, ln) for pn in selected_proteins for ln in selected_ligands]
    if not pairs:
        print('No pairs selected.'); return []
    print(f'Docking {len(pairs)} pair(s)...')
    p_bar = dad_progress('Phase B Docking', max(1, len(pairs)))
    results = []
    for pname, lname in pairs:
        result = run_gnina(pname, lname)
        results.append(result)
        p_bar.update(f'{result.case_id}: {result.status}')
    p_bar.close()
    return results

# Wire widget run button
if '_ui_ready' in globals() and _ui_ready:
    def _on_run_click(b):
        global docking_results
        with out_widget:
            out_widget.clear_output()
            docking_results = _run_selected_docking()
    run_btn.on_click(_on_run_click)
    print('Run button wired. Click "Run docking on selection" in SS6.')
    print('Or call _run_selected_docking() directly:')
    print('  docking_results = _run_selected_docking()')
else:
    docking_results = _run_selected_docking()

In [ ]:
#@title 8. Phase B — Aggregate Results {display-mode: "form"}
#@markdown Appends completed pairs to `docking_master.csv` (append-only, dedup on case_id).
#@markdown **Output:** `docking_master.csv`, ranked table, protein x ligand CNN score heatmap.

import pandas as pd, matplotlib.pyplot as plt, numpy as np

MASTER_CSV = PHASE_B_DIR / 'docking_master.csv'

p = dad_progress('Phase B Aggregate', 4)

if 'docking_results' not in globals() or not docking_results:
    print('No docking results. Run SS7 first (click "Run docking on selection" or call _run_selected_docking()).')
else:
    new_rows = [{
        'case_id': r.case_id, 'protein_name': r.protein_name, 'ligand_name': r.ligand_name,
        'status': r.status, 'vina_score': r.vina_score, 'cnn_pose_score': r.cnn_pose_score,
        'cnn_affinity': r.cnn_affinity, 'pose_count': r.pose_count,
        'runtime_s': r.runtime_s, 'out_sdf': r.out_sdf,
    } for r in docking_results]
    new_df = pd.DataFrame(new_rows)
    p.update('build rows')

    combined = (pd.concat([pd.read_csv(MASTER_CSV), new_df]).drop_duplicates(subset='case_id', keep='last')
                if MASTER_CSV.exists() else new_df)
    combined.to_csv(MASTER_CSV, index=False)
    p.update('write master CSV')

    ok_df = new_df[new_df['status'].isin(['ok', 'cache_hit'])].copy()
    for col in ['vina_score', 'cnn_pose_score', 'cnn_affinity', 'runtime_s']:
        ok_df[col] = pd.to_numeric(ok_df[col], errors='coerce')
    ok_df = ok_df.sort_values(['cnn_pose_score', 'cnn_affinity', 'vina_score'],
                               ascending=[False, False, True])
    ok_df.insert(0, 'rank', range(1, len(ok_df) + 1))
    display_cols = ['rank', 'case_id', 'status', 'cnn_pose_score', 'cnn_affinity',
                    'vina_score', 'pose_count', 'runtime_s']
    try:
        from IPython.display import display as _d; _d(ok_df[display_cols])
    except Exception: print(ok_df[display_cols].to_string())
    p.update('ranked table')

    if len(ok_df) > 1:
        try:
            pivot = ok_df.pivot_table(index='protein_name', columns='ligand_name',
                                       values='cnn_pose_score', aggfunc='max')
            fig, ax = plt.subplots(figsize=(max(4, len(pivot.columns) * 1.2),
                                             max(3, len(pivot.index) * 0.8)))
            im = ax.imshow(pivot.values, aspect='auto', cmap='viridis', vmin=0, vmax=1)
            ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(list(pivot.columns), rotation=45, ha='right')
            ax.set_yticks(range(len(pivot.index)));   ax.set_yticklabels(list(pivot.index))
            ax.set_title('Protein x Ligand CNN pose score heatmap (diagnostic)')
            fig.colorbar(im, ax=ax, label='CNN pose score'); plt.tight_layout(); plt.show()
        except Exception as e: print(f'Heatmap skipped: {e}')
    p.update('heatmap'); p.close()

    print(f'docking_master.csv: {MASTER_CSV} ({len(combined)} total pairs)')

In [ ]:
#@title 9. Phase B — Structure Visualization {display-mode: "form"}
#@markdown py3Dmol views for top-ranked docked pairs.
#@markdown **Output:** interactive 3D viewer (protein cartoon + ligand sticks).

MAX_VIEW_CASES = 3  #@param {type:"integer"}

p = dad_progress('Visualization', max(1, MAX_VIEW_CASES))
if 'docking_results' not in globals() or not docking_results:
    print('No docking results. Run SS7 first.')
else:
    try:
        import py3Dmol
        ok = sorted([r for r in docking_results if r.status in ('ok','cache_hit') and r.out_sdf],
                    key=lambda r: r.cnn_pose_score or -1.0, reverse=True)
        for result in ok[:MAX_VIEW_CASES]:
            pdb_path = structure_registry.get(result.protein_name, {}).get('pdb_path', '')
            if not pdb_path or not Path(pdb_path).exists():
                print(f'  {result.case_id}: missing PDB'); p.update(result.case_id); continue
            view = py3Dmol.view(width=800, height=500)
            view.addModel(Path(pdb_path).read_text(), 'pdb')
            view.setStyle({'model': 0}, {'cartoon': {'color': 'spectrum'}})
            if Path(result.out_sdf).exists():
                view.addModel(Path(result.out_sdf).read_text(), 'sdf')
                view.setStyle({'model': 1}, {'stick': {'colorscheme': 'greenCarbon'}})
            view.zoomTo(); view.show()
            print(f'  {result.case_id}: CNN_pose={result.cnn_pose_score}, '
                  f'CNN_affinity={result.cnn_affinity}, Vina={result.vina_score}')
            p.update(result.case_id)
    except ImportError:
        print('py3Dmol not available.')
p.close()

In [ ]:
#@title 10. Reproducibility Footprint {display-mode: "form"}
#@markdown Records Phase A + Phase B SHA256 hashes, tool versions, GPU info.
#@markdown **Output:** `manifest.json` in WORK_DIR.

import platform

p = dad_progress('Reproducibility', 4)

footprint: dict = {
    'exec_mode': 'colab' if IN_COLAB else 'local',
    'python_version': platform.python_version(),
    'platform': platform.platform(),
    'structure_mode': STRUCTURE_MODE,
    'af2_fast_preview': AF2_FAST_PREVIEW if STRUCTURE_MODE in ('colabfold_af2', 'auto') else None,
    'n_proteins': len(structure_registry),
    'n_ligands': len(ligand_registry) if 'ligand_registry' in globals() else 0,
    'n_docking_results': len(docking_results) if 'docking_results' in globals() else 0,
    'structure_registry_sha': {k: v['seq_sha'] for k, v in structure_registry.items()},
    'ligand_registry_sha': {
        k: v.smiles_sha for k, v in ligand_registry.items()
    } if 'ligand_registry' in globals() else {},
}
p.update('hashes')

pkg_versions: dict = {}
for pkg in ['rdkit', 'pandas', 'numpy', 'matplotlib', 'scipy', 'scikit-learn', 'biopython', 'py3Dmol']:
    try:
        import importlib.metadata; pkg_versions[pkg] = importlib.metadata.version(pkg)
    except Exception: pkg_versions[pkg] = 'not found'
footprint['packages'] = pkg_versions
p.update('packages')

try:
    r = subprocess.run([str(GNINA_BIN), '--version'], capture_output=True, text=True, timeout=5)
    footprint['gnina_version'] = (r.stdout + r.stderr).strip()[:200]
except Exception:
    footprint['gnina_version'] = 'not available'

try:
    gpu_r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                            capture_output=True, text=True, timeout=5)
    footprint['gpu'] = gpu_r.stdout.strip()[:200]
except Exception:
    footprint['gpu'] = 'not available'
p.update('GPU info')

manifest_path = WORK_DIR / 'manifest.json'
manifest_path.write_text(json.dumps(footprint, indent=2))
p.update('write manifest'); p.close()

print(json.dumps({k: footprint[k] for k in [
    'exec_mode', 'structure_mode', 'n_proteins', 'n_ligands',
    'n_docking_results', 'gnina_version', 'gpu'
]}, indent=2))
print(f'Full manifest: {manifest_path}')

---
### Structure Prediction and Docking — References

- **AlphaFold 3 (optional ingest):** Abramson J et al. (2024) Accurate structure prediction of biomolecular interactions with AlphaFold 3. *Nature* 630:493-500. https://doi.org/10.1038/s41586-024-07487-w  
  *Note: AF3 weights are restricted by DeepMind. This notebook ingests pre-computed AF3 results only; direct AF3 execution on Colab is not supported.*
- **AlphaFold 2 / ColabFold (default):** Mirdita M et al. (2022) ColabFold: making protein folding accessible to all. *Nat Methods* 19:679-682. https://doi.org/10.1038/s41592-022-01488-1 | Jumper J et al. (2021) *Nature* 596:583-589.
- **ESMFold (auto fallback):** Lin Z et al. (2023) Evolutionary-scale prediction of atomic-level protein structure with a language model. *Science* 379:1123-1130. https://doi.org/10.1126/science.ade2574
- **GNINA:** McNutt AT et al. (2021) GNINA 1.0: molecular docking with deep learning. *J Cheminform* 13:43. https://doi.org/10.1186/s13321-021-00522-2
- **P2Rank:** Krivak R & Hoksza D (2018) P2Rank: machine learning based tool for rapid and accurate prediction of ligand binding sites. *J Cheminform* 10:39.